# Parallel Execution: Practice Exercise (Solution)

Build a parallel product review analyzer that processes multiple product reviews concurrently using LangGraph's Send API. You will apply the same parallelization pattern from the lesson to a different domain.

**What you'll implement:**
- A router function that creates Send objects for parallel processing
- A review analysis node that processes individual reviews
- State configuration for aggregating parallel results

**Estimated time:** 12-15 minutes

## Setup

Run this cell to import all required libraries and initialize the API client.

In [1]:
# Setup - run this cell first

import os
import json
import time
from typing import TypedDict, List, Dict, Any, Annotated
from operator import add

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Setup complete!")

Setup complete!


## Context

You are building a product review analysis system for an e-commerce platform. The system receives a batch of customer reviews and needs to analyze each one for:
- Sentiment (positive, negative, neutral)
- Key themes mentioned
- An overall quality score (1-5)

**The Problem:** Analyzing reviews sequentially is slow. If you have 4 reviews and each takes 3 seconds to analyze, sequential processing takes 12 seconds. With parallel execution, all 4 can be analyzed simultaneously in ~3 seconds.

**Your Task:** Implement parallel review analysis using LangGraph's Send API pattern:
1. A router creates `Send` objects, one per review
2. Each `Send` triggers an independent analysis node
3. Results are automatically aggregated using `Annotated[List, add]`

**Input:** A list of product review strings

**Output:** A list of analysis results, one per review, containing sentiment, themes, and score

## Sample Data

Here are the reviews your system will analyze.

In [2]:
# Sample product reviews to analyze
sample_reviews = [
    "Absolutely love this wireless keyboard! The keys are responsive and quiet. Battery life is incredible - haven't charged it in 3 weeks. Only minor complaint is the Bluetooth pairing can be finicky sometimes.",
    "Terrible experience. The product arrived damaged and customer service was unhelpful. Took 2 weeks to get a refund. Would not recommend to anyone.",
    "It's okay for the price. Does what it says but nothing special. Build quality feels a bit cheap but it works fine for basic use.",
    "Game changer for my home office! Setup was easy, the ergonomic design has helped my wrist pain, and it looks sleek on my desk. Worth every penny."
]

print(f"Number of reviews to analyze: {len(sample_reviews)}")
for i, review in enumerate(sample_reviews, 1):
    print(f"\nReview {i}: {review[:60]}...")

Number of reviews to analyze: 4

Review 1: Absolutely love this wireless keyboard! The keys are respons...

Review 2: Terrible experience. The product arrived damaged and custome...

Review 3: It's okay for the price. Does what it says but nothing speci...

Review 4: Game changer for my home office! Setup was easy, the ergonom...


## Pydantic Model for Structured Output

This model defines the structure of each review analysis result.

In [3]:
class ReviewAnalysis(BaseModel):
    """Structured analysis of a product review."""
    sentiment: str = Field(description="Overall sentiment: 'positive', 'negative', or 'neutral'")
    themes: List[str] = Field(description="Key themes mentioned in the review (e.g., 'battery life', 'customer service')")
    score: int = Field(description="Quality score from 1 (very negative) to 5 (very positive)", ge=1, le=5)
    summary: str = Field(description="One-sentence summary of the review")


print("ReviewAnalysis model defined!")

ReviewAnalysis model defined!


## Part 1: Define the State Schema

Define the workflow state. The key insight is using `Annotated[List, add]` for the results field so that parallel outputs are automatically accumulated.

In [4]:
# SOLUTION: Define the state schema with Annotated for parallel accumulation

class ReviewWorkflowState(TypedDict):
    """
    State for the review analysis workflow.
    
    Attributes:
        reviews: List of review strings to analyze
        analysis_results: Accumulated results from parallel analysis
                         Uses Annotated[List, add] to merge parallel outputs
    """
    reviews: List[str]
    analysis_results: Annotated[List[Dict[str, Any]], add]


print("State schema defined!")

State schema defined!


## Part 2: Create the Router Function

The router examines the state and returns a list of `Send` objects. Each `Send` tells LangGraph to invoke a specific node with specific data. When you return multiple `Send` objects, LangGraph executes them in parallel.

In [5]:
# SOLUTION: Router function that creates Send objects for parallel processing

def route_reviews_for_analysis(state: ReviewWorkflowState):
    """
    Router that creates parallel Send commands for each review.
    
    This function is called after the start node. It should:
    1. Get the list of reviews from state
    2. Create a Send object for each review
    3. Return the list of Send objects
    
    Each Send should:
    - Target the "analyze_review" node
    - Pass a dict with "review_text" and "review_index" keys
    
    Args:
        state: The current workflow state containing reviews
        
    Returns:
        List[Send]: One Send object per review for parallel execution
    """
    reviews = state["reviews"]
    
    print(f"Routing {len(reviews)} reviews for parallel analysis...")
    
    return [
        Send("analyze_review", {"review_text": review, "review_index": i})
        for i, review in enumerate(reviews, 1)
    ]


print("Router function defined!")

Router function defined!


## Part 3: Create the Analysis Node

This node receives data from a `Send` object and analyzes a single review. It will be invoked multiple times in parallel (once per review).

In [6]:
# SOLUTION: Analysis node that processes individual reviews

def analyze_review_node(review_data: dict) -> dict:
    """
    Analyzes a single review using the LLM.
    
    This node is invoked in parallel by Send objects from the router.
    Each invocation processes one review independently.
    
    Args:
        review_data: Dict containing:
            - review_text (str): The review to analyze
            - review_index (int): The index of this review (for logging)
    
    Returns:
        dict: Must return {"analysis_results": [result_dict]}
              The list wrapper is required for Annotated[List, add] accumulation
    """
    review_text = review_data["review_text"]
    review_index = review_data["review_index"]
    
    print(f"[Review {review_index}] Starting analysis...")
    
    system_prompt = """
    You are a product review analyst. Analyze the given review and extract:
    - Sentiment: 'positive', 'negative', or 'neutral'
    - Themes: Key topics mentioned (e.g., 'battery life', 'build quality', 'customer service')
    - Score: 1-5 rating based on overall sentiment (1=very negative, 5=very positive)
    - Summary: One sentence summarizing the review
    
    Be objective and thorough in your analysis.
    """
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Analyze this review:\n\n{review_text}")
    ]
    
    analyzer = llm.with_structured_output(ReviewAnalysis)
    result = analyzer.invoke(messages)
    
    print(f"[Review {review_index}] Analysis complete - Sentiment: {result.sentiment}")
    
    # Return in format that enables accumulation via Annotated[List, add]
    return {"analysis_results": [result.model_dump()]}


print("Analysis node defined!")

Analysis node defined!


## Part 4: Create the Aggregation Node

This node runs after all parallel analyses complete. Due to `Annotated[List, add]`, results are already accumulated in state.

In [7]:
# SOLUTION: Aggregation node (results already accumulated by LangGraph)

def aggregate_results_node(state: ReviewWorkflowState) -> ReviewWorkflowState:
    """
    Aggregates results after parallel processing completes.
    
    Because we used Annotated[List, add] in the state schema,
    LangGraph automatically accumulated all parallel results.
    This node just logs completion and passes state through.
    
    Args:
        state: The workflow state with accumulated analysis_results
        
    Returns:
        The state (unchanged, results already accumulated)
    """
    results = state["analysis_results"]
    print(f"\nAll {len(results)} reviews analyzed!")
    print("Results automatically aggregated by LangGraph.")
    return state


print("Aggregation node defined!")

Aggregation node defined!


## Part 5: Build the Workflow Graph

Assemble the nodes into a graph. The key is using `add_conditional_edges` with the router function to create parallel branches.

In [8]:
# SOLUTION: Build the complete workflow graph

workflow_builder = StateGraph(ReviewWorkflowState)

# Add nodes
workflow_builder.add_node("analyze_review", analyze_review_node)
workflow_builder.add_node("aggregate_results", aggregate_results_node)

# Add conditional edges from START using the router
# This creates parallel branches - one Send per review
workflow_builder.add_conditional_edges(START, route_reviews_for_analysis)

# All parallel branches converge at aggregate_results
workflow_builder.add_edge("analyze_review", "aggregate_results")

# Final edge to END
workflow_builder.add_edge("aggregate_results", END)

# Compile the workflow
review_analyzer = workflow_builder.compile()

print("Review analyzer workflow built!")

Review analyzer workflow built!


## Run Your Implementation

Test your parallel review analyzer and observe the execution timing.

In [10]:
# Run the review analyzer
print("="*60)
print("PARALLEL REVIEW ANALYSIS")
print("="*60)

initial_state = {
    "reviews": sample_reviews,
    "analysis_results": []
}

# Measure execution time
start_time = time.time()
result = review_analyzer.invoke(initial_state)
elapsed = time.time() - start_time

print(f"\nTotal execution time: {elapsed:.2f} seconds")
print(f"Reviews analyzed: {len(result['analysis_results'])}")

# Display results
print("\n" + "="*60)
print("ANALYSIS RESULTS")
print("="*60)

for i, analysis in enumerate(result['analysis_results'], 1):
    print(f"\nReview {i}:")
    print(f"  Sentiment: {analysis['sentiment']}")
    print(f"  Score: {analysis['score']}/5")
    print(f"  Themes: {', '.join(analysis['themes'])}")
    print(f"  Summary: {analysis['summary']}")

PARALLEL REVIEW ANALYSIS
Routing 4 reviews for parallel analysis...
[Review 1] Starting analysis...
[Review 2] Starting analysis...
[Review 3] Starting analysis...
[Review 4] Starting analysis...
[Review 3] Analysis complete - Sentiment: neutral
[Review 1] Analysis complete - Sentiment: positive
[Review 2] Analysis complete - Sentiment: negative
[Review 4] Analysis complete - Sentiment: positive

All 4 reviews analyzed!
Results automatically aggregated by LangGraph.

Total execution time: 2.50 seconds
Reviews analyzed: 8

ANALYSIS RESULTS

Review 1:
  Sentiment: positive
  Score: 4/5
  Themes: keyboard performance, battery life, Bluetooth connectivity
  Summary: The reviewer loves the wireless keyboard for its responsive keys and excellent battery life, though notes occasional issues with Bluetooth pairing.

Review 2:
  Sentiment: negative
  Score: 1/5
  Themes: product quality, customer service, refund process
  Summary: The review expresses dissatisfaction due to a damaged product an